In [1]:
import polars as pl
import polars.selectors as cs
from hta.trace_analysis import TraceAnalysis
import pandas as pd
import glob
import os

In [2]:
dfs = []
for path in glob.glob("/proj/threadtune-PG0/amir/kineto_traces/*"):
    analyzer = TraceAnalysis({0: "device.0.json"}, path)
    rank = 0
    trace_data = analyzer.t.get_trace(rank)
    symbol_table = analyzer.t.symbol_table.get_sym_table()
    dfs.append(
        pl.concat(
            pl.from_pandas(trace_data[trace_data["stream"] != -1])
            .select(pl.lit(rank).alias("rank"), pl.all())
            for rank, trace_data in analyzer.t.traces.items()
        ).join(
            pl.from_dict({"name": list(range(len(symbol_table))), "s_name": symbol_table}),
            on="name",
        )
    )
dfs

Parsed /proj/threadtune-PG0/amir/kineto_traces/deit-train/device.0.json time = 0.20 seconds 
Rounding down ns resolution events due to issue with events overlapping. ts dtype = float64, dur dtype = float64.Please see https://github.com/pytorch/pytorch/pull/122425
Parsed /proj/threadtune-PG0/amir/kineto_traces/deit-train/device.0.json backend=json in 0.47 seconds; current PID:572824
Overall parsing of /proj/threadtune-PG0/amir/kineto_traces/deit-train/device.0.json in 0.53 seconds; current PID:572824
leaving parse_multiple_ranks duration=0.54 seconds
leaving parse_traces duration=0.54 seconds
There is only one iteration in the trace. The analysis result may not be accurate.
Parsed /proj/threadtune-PG0/amir/kineto_traces/gpt2-inference/device.0.json time = 0.10 seconds 
Rounding down ns resolution events due to issue with events overlapping. ts dtype = float64, dur dtype = float64.Please see https://github.com/pytorch/pytorch/pull/122425
Parsed /proj/threadtune-PG0/amir/kineto_traces/gpt

[shape: (1_131, 22)
 ┌──────┬───────┬─────┬──────┬───┬─────────────┬───────────────────┬───────────┬────────────────────┐
 │ rank ┆ index ┆ cat ┆ name ┆ … ┆ external_id ┆ index_correlation ┆ iteration ┆ s_name             │
 │ ---  ┆ ---   ┆ --- ┆ ---  ┆   ┆ ---         ┆ ---               ┆ ---       ┆ ---                │
 │ i32  ┆ i16   ┆ i64 ┆ i64  ┆   ┆ i16         ┆ i16               ┆ i8        ┆ str                │
 ╞══════╪═══════╪═════╪══════╪═══╪═════════════╪═══════════════════╪═══════════╪════════════════════╡
 │ 0    ┆ 13921 ┆ 68  ┆ 184  ┆ … ┆ 7           ┆ 13923             ┆ 0         ┆ Memset (Device)    │
 │ 0    ┆ 13938 ┆ 97  ┆ 163  ┆ … ┆ 7           ┆ 13940             ┆ 0         ┆ void implicit_conv │
 │      ┆       ┆     ┆      ┆   ┆             ┆                   ┆           ┆ olve_sgemm<f…      │
 │ 0    ┆ 13945 ┆ 97  ┆ 114  ┆ … ┆ 10          ┆ 13947             ┆ 0         ┆ void at::native::e │
 │      ┆       ┆     ┆      ┆   ┆             ┆              

In [39]:
analyzer = TraceAnalysis({0: "device.0.json"}, "/proj/threadtune-PG0/amir/kineto_traces/bert-inference/")
rank = 0
trace_data = analyzer.t.get_trace(rank)
symbol_table = analyzer.t.symbol_table.get_sym_table()
trace_df = pl.concat(
    pl.from_pandas(trace_data[trace_data["stream"] != -1][["name", "dur"]])
    .select(pl.lit(rank).alias("rank"), pl.all())
    for rank, trace_data in analyzer.t.traces.items()
).join(
    pl.from_dict({"name": list(range(len(symbol_table))), "s_name": symbol_table}),
    on="name",
).filter(
    ~pl.col("s_name").str.starts_with("nccl") & 
    ~pl.col("s_name").is_in([
        "Stream Sync",
        "Memcpy DtoD (Device -> Device)",
        "Memcpy HtoD (Pageable -> Device)",
        "Memcpy DtoH (Device -> Pinned)",
    ])
).select(pl.lit("bert").alias("model"), pl.all())
trace_df

Parsed /proj/threadtune-PG0/amir/kineto_traces/bert-inference/device.0.json time = 0.04 seconds 
Rounding down ns resolution events due to issue with events overlapping. ts dtype = float64, dur dtype = float64.Please see https://github.com/pytorch/pytorch/pull/122425
Parsed /proj/threadtune-PG0/amir/kineto_traces/bert-inference/device.0.json backend=json in 0.13 seconds; current PID:572824
Overall parsing of /proj/threadtune-PG0/amir/kineto_traces/bert-inference/device.0.json in 0.15 seconds; current PID:572824
leaving parse_multiple_ranks duration=0.16 seconds
leaving parse_traces duration=0.16 seconds
There is only one iteration in the trace. The analysis result may not be accurate.


model,rank,name,dur,s_name
str,i32,i64,f64,str
"""bert""",0,59,9.0,"""void at::native::(anonymous na…"
"""bert""",0,59,4.0,"""void at::native::(anonymous na…"
"""bert""",0,73,1.0,"""void at::native::vectorized_el…"
"""bert""",0,59,7.0,"""void at::native::(anonymous na…"
"""bert""",0,73,0.0,"""void at::native::vectorized_el…"
…,…,…,…,…
"""bert""",0,69,7.0,"""void gemv2T_kernel_val<int, in…"
"""bert""",0,52,2.0,"""void at::native::vectorized_el…"
"""bert""",0,7,3.0,"""void at::native::(anonymous na…"


In [40]:
accel_df = pl.read_csv(
    "/users/zarand1a/accel-sim-framework/kernel_times.csv",
).select(
    pl.col("benchmark").alias("model"),
    pl.col("kernel_demangled").str.strip_chars().alias("kernel"),
    (pl.col("cycle") / 1132).alias("accel_time_us"),
)

In [47]:
accel_df.filter(pl.col("model") == "bert").write_csv("accel_time.csv")

In [26]:
accel_df.filter(pl.col("model") == "bert")

model,kernel,accel_time_us
str,str,f64
"""bert""","""void at::native::<unnamed>::in…",9.166078
"""bert""","""void at::native::<unnamed>::in…",7.567138
"""bert""","""void at::native::vectorized_el…",5.09894
"""bert""","""void at::native::<unnamed>::in…",9.189046
"""bert""","""void at::native::vectorized_el…",5.09894
…,…,…
"""bert""","""void at::native::vectorized_el…",4.864841
"""bert""","""void at::native::vectorized_el…",5.137809
"""bert""","""void at::native::vectorized_el…",4.948763


In [46]:
(trace_data["cat"] == 33).sum()

np.int64(1327)

In [21]:
accel_df.join(
    trace_df.select("model", pl.col("s_name").alias("kernel"), pl.col("dur").alias("trace_time_us")),
    on=["model", "kernel"],
)

model,kernel,accel_time_us,trace_time_us
str,str,f64,f64


In [49]:
trace_df.select("model", pl.col("s_name").alias("kernel"), pl.col("dur").alias("trace_time_us")).write_csv("trace_time.csv")